## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [7]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [8]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

## Feature Engineering - 1st Digit and 2 first Digit - Benford's Law

In [9]:
def extract_first_digit_features(df, numeric_cols,n=1):
    """Extracts the leading digits (e.g., 1st digit or first 2 digits)

    from numeric columns for Benford's Law analysis.

    Parameters:
    - df: pandas DataFrame
    - numeric_cols: list of columns to process
    - n_digits: number of leading digits to extract (default is 1)
    """
    df = df.copy()

    for col in numeric_cols:
        # 1. Convert to string, strip all non-digit characters (decimals, signs)
        str_series = (
            df[col].astype(str).str.replace(r"\D+", "", regex=True).str.lstrip("0")
        )

        # 2. Extract the first n digits (handling empty strings where values were 0)
        feature_name = (
            f"{col}_first_digit"
        )

        # 3. Extract the first n characters, handle blanks as 'Missing' or NaN,
        # and convert the result into a categorical type
        extracted = (
            str_series.str[:n].replace("", np.nan).fillna("Missing")
        )  # Using string 'Missing' to keep categories clean

        # Extract slice and handle cases where string length is less than n_digits
        df[feature_name] = extracted.astype("category")

    return df

In [10]:
def extract_first_two_digits(df, numeric_cols):
  """Extracts the first two leading digits from numeric columns.

  Handles negatives, decimals, and leading zeros (e.g., 0.0034 -> 34).

  Parameters:
  - df: pandas DataFrame
  - numeric_cols: list of column names to process

  Returns:
  - DataFrame with new columns named '{col}_first_two_digits'
  """
  df = df.copy()

  for col in numeric_cols:
    # 1. Convert to string and remove non-digit characters (signs, decimals, exponents)
    # Then strip any leading zeros
    str_series = (
        df[col].astype(str).str.replace(r"[^\d]", "", regex=True).str.lstrip("0")
    )

    # 2. Extract the first 2 characters, handling empty strings (e.g., original value was 0)
    extracted = str_series.str[:2]

    # 3. Format into a clean numeric feature (using -1 for missing/zero values)
    feature_name = f"{col}_first_two_digits"
    df[feature_name] = (
        extracted.replace("", np.nan).fillna("Missing").astype("category")
    )

  return df

In [11]:
numeric_features = ["amount_usd", "account_balance_usd"]

In [12]:
train_featured = extract_first_digit_features(train, numeric_features)

In [13]:
train_featured = extract_first_two_digits(train_featured, numeric_features)

In [14]:
test_featured = extract_first_digit_features(test, numeric_features)

In [15]:
test_featured = extract_first_two_digits(test_featured, numeric_features)

In [16]:
cat_cols = train_featured.select_dtypes(include='category').columns.tolist()

In [17]:
category_cols = obj_cols + cat_cols

## Building TE Target Encoding with Stratified KFOLD

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [20]:
X = train_featured.drop(columns=["is_fraud"]).copy()
y = train_featured["is_fraud"].copy()

In [21]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [22]:
FOLDS

[(array([     0,      1,      2, ..., 699956, 699958, 699959]),
  array([     8,     10,     16, ..., 699945, 699948, 699957])),
 (array([     0,      3,      5, ..., 699957, 699958, 699959]),
  array([     1,      2,      4, ..., 699951, 699952, 699956])),
 (array([     1,      2,      3, ..., 699957, 699958, 699959]),
  array([     0,     19,     24, ..., 699944, 699949, 699954])),
 (array([     0,      1,      2, ..., 699957, 699958, 699959]),
  array([    13,     15,     18, ..., 699942, 699950, 699955])),
 (array([     0,      1,      2, ..., 699955, 699956, 699957]),
  array([     3,      5,     11, ..., 699953, 699958, 699959]))]

In [30]:
len(FOLDS[1][0])

559968

In [ ]:
def run_cv(te_cols): # adding input te_cols for categorical variable required for TE
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)
        
    return oof, test_pred, total

## XGboost + Optuna

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [28]:
from sklearn.preprocessing import StandardScaler

In [29]:
import xgboost as xgb

In [30]:
X = train_featured.drop(columns=["id","is_fraud"]).copy()
y = train_featured["is_fraud"].copy()

### Insert TE in the run_cv loop

In [41]:
def run_cv(fit_predict_fn, te_cols, name): # adding input te_cols for categorical variable required for TE
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        # --- FIX: Fill NaNs and ensure string type for TE columns ---
        #for col in te_cols:
        #    X_tr.loc[:,col] = X_tr.loc[:,col].fillna("Missing").astype(str)
        #   X_va.loc[:,col] = X_va.loc[:,col].fillna("Missing").astype(str)
        #   X_test.loc[:,col] = X_test.loc[:,col].fillna("Missing").astype(str)
        
        # -----------------------------------------------------------
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])
        
        # transform on X_test data
        X_test = test_featured.drop(columns=["id"]).copy()
        # X_test_obj = X_test[obj_cols].copy()
        # print(X_test.info())
        X_test.loc[:, te_cols] = te.transform(X_test[te_cols])

        # print(f'x_test',X_test.head(2))

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
            # X_tr.loc[:,col] = X_tr.loc[:,col].astype('float64')
            # X_va.loc[:,col] = X_va.loc[:,col].astype('float64')
            # X_test.loc[:,col] = X_test.loc[:,col].astype('float64')

        # print(f'x_tr ', X_tr.info())
        # print(f'x_va ', X_va.info())
        # print(f'x_test ', X_test.info())
        # start predit
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)

        # save prediction in oof and in test
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [44]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))